# Gemini Text-to-SQL Chatbot

This notebook converts natural-language questions into SQL using Gemini and LangChain, connects to PostgreSQL, and evaluates generated SQL with RAGAS.

## Security
- API keys and database passwords are loaded from environment variables.
- Do **not** commit `.env` to GitHub.
- Database query outputs have been cleared before publishing this notebook.


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
DB_USER = os.getenv("DB_USER", "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "text_to_sql")

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY is not set. Add it to your local .env file.")
if not DB_PASSWORD:
    raise ValueError("DB_PASSWORD is not set. Add it to your local .env file.")


In [ ]:
%pip install -U langchain langchain-community langchain-google-genai langchain-core

In [ ]:
from langchain_community.utilities import SQLDatabase
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [ ]:
%pip install -U langchain langchain-community langchain-core langchain-google-genai sqlalchemy psycopg2-binary

In [ ]:
from sqlalchemy import create_engine, text
from langchain_community.utilities import SQLDatabase
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
DB_USER = "postgres"
DB_PASSWORD = ""
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "text_to_sql"

connection_string = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(connection_string)

In [ ]:
try:
    with engine.connect() as connection:
        result = connection.execute(text("SELECT version();"))
        print("PostgreSQL Connected Successfully! ✅")
        print(result.fetchone()[0])

except Exception as e:
    print("Connection Failed ❌")
    print(e)

In [ ]:
db = SQLDatabase.from_uri(connection_string)

print("Database connected to LangChain successfully! ✅")

In [ ]:
tables = db.get_usable_table_names()

print("Tables in database:")
print(tables)

In [ ]:
context = db.get_table_info()

print(context)

In [ ]:
# Create the LLM Prompt Template                  
from langchain_core.prompts import ChatPromptTemplate

template = """Based on the table schema below, write a SQL query that would answer the user's question:
Remember : Only provide me the sql query dont include anything else.
           Provide me sql query in a single line dont add line breaks.
Table Schema:
{schema}

Question: {question}
SQL Query:
"""
prompt = ChatPromptTemplate.from_template(template)

In [ ]:
# get the schema of the database
def get_schema(db):
    schema = db.get_table_info()
    return schema

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    api_key=os.getenv("GEMINI_API_KEY")
)

In [ ]:
# Create the SQL query chain using the LLM and the prompt template
sql_chain = (
    RunnablePassthrough.assign(schema=lambda _: get_schema(db))
    | prompt
    | llm.bind(stop=["\nSQLResult:"])
    | StrOutputParser()
)

In [ ]:
#test the SQL query chain with a sample question
resp=sql_chain.invoke({"question":"List the top 5 customers who spent the highest amount overall and give amounta also"})
print(resp)

In [ ]:
query = resp.strip()

print("QUERY:", query)
print("TYPE:", type(query))

result = db.run(query)
print(result)

In [ ]:
import pandas as pd

# Direct Pandas Table ke roop mein load karein
df_result = pd.read_sql(query, db._engine)

# Clean Table display karein
display(df_result)

In [ ]:
%pip install "langchain-community==0.4.1" --force-reinstall --no-deps

In [ ]:
%pip install -U ragas

In [ ]:
# STEP 3 — RAGAS core imports
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import ContextPrecision, Faithfulness, RubricsScore
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset
from ragas import evaluate

In [ ]:
# STEP 4 — Evaluator LLM: reuse the SAME Gemini `llm` object you created earlier
# (no need for a separate Groq / OpenAI key — this avoids another whole set of dependency problems)
evaluator_llm = LangchainLLMWrapper(llm)

In [ ]:
# STEP 5 — Define the metrics
# ContextPrecision and Faithfulness only need an LLM (no embeddings needed -> no HuggingFace download)
context_precision = ContextPrecision(llm=evaluator_llm)
faithfulness = Faithfulness(llm=evaluator_llm)

helpfulness_rubrics = {
    "score1_description": "The generated SQL is unrelated to the question or would fail to run.",
    "score2_description": "The SQL touches the right table but answers the wrong question or is missing key filters/joins.",
    "score3_description": "The SQL mostly answers the question but has minor issues (wrong column, missing ORDER BY/LIMIT, etc).",
    "score4_description": "The SQL correctly answers the question, with only very minor stylistic differences from the ideal query.",
    "score5_description": "The SQL exactly and efficiently answers the question, matching the reference query's intent.",
}
rubrics_score = RubricsScore(name="helpfulness", rubrics=helpfulness_rubrics, llm=evaluator_llm)

In [ ]:
schema_context = db.get_table_info()

user_inputs = [
    "List the top 5 customers who spent the highest amount overall and give amount also",
    "What is the 2017 budget for Product 5?",
    "List all customer names from the customers table.",
]

# ground-truth SQL for each question above — EDIT these to match what you consider the 'correct' query
references = [
    "SELECT customer_names, SUM(line_total) AS total_spent FROM customers JOIN sales_order ON customers.customer_index = sales_order.customer_name_index GROUP BY customer_names ORDER BY total_spent DESC LIMIT 5;",
    "SELECT budget_2017 FROM budgets_2017 WHERE product_name = 'Product 5';",
    "SELECT customer_names FROM customers;",]

In [ ]:
responses = []
for q in user_inputs:
    resp = sql_chain.invoke({"question": q})
    responses.append(resp.strip())

for q, r in zip(user_inputs, responses):
    print(f"Q: {q}\nA: {r}\n")

In [ ]:
# STEP 8 — Build the RAGAS EvaluationDataset
samples = []
for i in range(len(user_inputs)):
    samples.append(
        SingleTurnSample(
            user_input=user_inputs[i],
            response=responses[i],
            retrieved_contexts=[schema_context],
            reference=references[i],
        )
    )

ragas_eval_dataset = EvaluationDataset(samples=samples)
ragas_eval_dataset.to_pandas()

In [ ]:
# STEP 9 — Run the evaluation
result = evaluate(
    dataset=ragas_eval_dataset,
    metrics=[context_precision, faithfulness, rubrics_score],
)
result

In [ ]:
# STEP 10 — View as a table
result_df = result.to_pandas()
result_df